<h1 align="center"> Lingüística Computacional - MIARFID</h1>
<h1 align="center"> Trabajo final - Combinación de modelos</h1>

**Estudiantes:** Carlos Broch Cubedo · Ferran Artero Merino

# 0. Configuración

In [1]:
!pip -q install iterative-stratification

In [2]:
import os, re, ast, json, time, zipfile, warnings
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, hamming_loss, f1_score
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit, MultilabelStratifiedKFold
from google.colab import drive

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
SEED = 42
np.random.seed(SEED)

drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/LC_TrabajoFinal"
RUNS_DIR = os.path.join(DATA_DIR, "outputs", "transformers", "runs")   # modelos individuales
ENS_OUT = os.path.join(DATA_DIR, "outputs", "ensembles")
ENS_DIR = os.path.join(ENS_OUT, "runs")                                # combinaciones
os.makedirs(ENS_DIR, exist_ok=True)
TRAIN_FILE = os.path.join(DATA_DIR, "dataset_train.csv")
TEST_FILE = os.path.join(DATA_DIR, "dataset_test.csv")

Mounted at /content/drive


In [3]:
HOLDOUT_SIZE, N_FOLDS = 0.15, 3
MAX_LABELS = 4

MEJORES  = ["QW_A1_posweight", "B3_llrd", "SB_B1_lora"]       # mejor variante de Qwen3, SPECTER2 y SciBERT
SPECTER2 = ["B3_llrd", "A1_posweight", "E3_specter2", "A2_asl", "B1_lora"]
DENSOS   = ["KNN_specter2", "EMB_specter2_C0.1", "KNN_bge-large"]
COMBOS = {
    "specter2_svm": ["B3_llrd", "A1_posweight", "SVM_ref"],
    "top3":         MEJORES,
    "top3_svm":     MEJORES + ["SVM_ref"],
    "top6":         MEJORES + ["A1_posweight", "SVM_ref", "KNN_specter2"],
    "todo":         list(dict.fromkeys(MEJORES + SPECTER2 + ["SVM_ref"] + DENSOS)),
}
METHODS = ["media", "ponderada", "votos", "stack_clase", "stack_todas", "stack_lgbm"]

GREEDY_ITERS = 20
STACK_C_CLASE = 1.0
STACK_C_TODAS = 0.05
LGBM_PARAMS = dict(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=10,
                   subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0, verbose=-1)

# 1. Datos y partición
Los mismos artículos únicos, holdout y particiones que en los capítulos anteriores, para que las puntuaciones guardadas correspondan a sus filas.

In [4]:
CLASSES = ['cs.AI', 'cs.CL', 'cs.CR', 'cs.CV', 'cs.CY', 'cs.DC', 'cs.GR', 'cs.HC', 'cs.IR', 'cs.IT', 'cs.LG',
           'cs.MA', 'cs.MM', 'cs.NE', 'cs.RO', 'cs.SI', 'cs.SY', 'eess.IV', 'eess.SP', 'eess.SY', 'math.IT',
           'math.OC', 'math.ST', 'physics.chem-ph', 'stat.AP', 'stat.ME', 'stat.ML', 'stat.TH']
L = len(CLASSES)

def norm(s):
    s = str(s).lower()
    s = re.sub(r"\$[^$]*\$", " ", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

raw = pd.read_csv(TRAIN_FILE)
raw["labels"] = raw["terms"].map(ast.literal_eval)
raw["title_n"], raw["summary_n"] = raw["titles"].map(norm), raw["summaries"].map(norm)
data = (raw.groupby(["title_n", "summary_n"], sort=False)
           .agg(labels=("labels", lambda ls: sorted({l for x in ls for l in x})))
           .reset_index(drop=True))
mlb = MultiLabelBinarizer(classes=CLASSES)
Y = mlb.fit_transform(data["labels"])
N = len(data)
test = pd.read_csv(TEST_FILE)

ROOTS = ["cs.CV", "cs.LG", "stat.ML"]
ROOT_IDX = np.array([CLASSES.index(r) for r in ROOTS])
ALIAS_PAIRS = [(a, b) for i, a in enumerate(CLASSES) for b in CLASSES[i + 1:]
               if (Y[:, CLASSES.index(a)] == Y[:, CLASSES.index(b)]).all()]

dev_idx, ho_idx = next(MultilabelStratifiedShuffleSplit(n_splits=1, test_size=HOLDOUT_SIZE, random_state=SEED)
                       .split(np.zeros(N), Y))
Y_dev, Y_ho = Y[dev_idx], Y[ho_idx]
FOLDS = list(MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
             .split(np.zeros(len(dev_idx)), Y_dev))
FOLD_ID = np.empty(len(dev_idx), dtype=int)
for k, (_, va) in enumerate(FOLDS):
    FOLD_ID[va] = k

assert np.load(os.path.join(RUNS_DIR, "SVM_ref", "fold0.npz"))["S_va"].shape[0] == len(FOLDS[0][1])
print(f"Artículos únicos: {N} | dev: {len(dev_idx)} | holdout: {len(ho_idx)} | test: {len(test)}")

Artículos únicos: 32577 | dev: 27674 | holdout: 4903 | test: 4885


# 2. Funciones
### Umbrales, posproceso y evaluación
Las mismas de los capítulos anteriores. Un experimento (individual o combinación) guarda por partición sus puntuaciones de validación, holdout y test.

In [5]:
def compute_metrics(labels, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1, "precision": precision,
            "recall": recall, "hamming_loss": hamming_loss(labels, preds)}

def tie_alias(S):
    S = np.array(S, dtype=np.float32, copy=True)
    for a, b in ALIAS_PAIRS:
        i, j = CLASSES.index(a), CLASSES.index(b)
        S[:, i] = S[:, j] = (S[:, i] + S[:, j]) / 2
    return S

def tune_thresholds(Yt, S, n_grid=200):
    thr = np.zeros(S.shape[1])
    for j in range(S.shape[1]):
        y, s = Yt[:, j].astype(bool), S[:, j]
        cand = np.unique(np.r_[np.quantile(s, np.linspace(0, 1, n_grid)), 0.0])
        pred = s[None, :] >= cand[:, None]
        tp = (pred & y).sum(1)
        fp = (pred & ~y).sum(1)
        fn = y.sum() - tp
        thr[j] = cand[(2 * tp / np.maximum(2 * tp + fp + fn, 1)).argmax()]
    return thr

def crossfit_thresholds(Yt, S, group_id):
    T = np.zeros_like(S)
    for g in np.unique(group_id):
        m = group_id == g
        T[m] = tune_thresholds(Yt[~m], S[~m])
    return T

def decode_flat(S, T):
    M = S - T
    P = (M >= 0).astype(int)
    no_root = P[:, ROOT_IDX].sum(1) == 0
    best = ROOT_IDX[M[:, ROOT_IDX].argmax(1)]
    P[no_root, best[no_root]] = 1
    for i in np.where(P.sum(1) > MAX_LABELS)[0]:
        on = np.where(P[i])[0]
        P[i, on[np.argsort(-M[i, on])[MAX_LABELS:]]] = 0
    for a, b in ALIAS_PAIRS:
        i, j = CLASSES.index(a), CLASSES.index(b)
        P[:, i] = P[:, j] = np.maximum(P[:, i], P[:, j])
    return P

In [6]:
def run_dir(name):
    e = os.path.join(ENS_DIR, name)
    return e if os.path.isdir(e) else os.path.join(RUNS_DIR, name)

def fold_path(name, k):
    return os.path.join(run_dir(name), f"fold{k}.npz")

def save_fold(name, k, S_va, S_ho, S_te, secs):
    np.savez(fold_path(name, k), S_va=S_va.astype(np.float32), S_ho=S_ho.astype(np.float32),
             S_te=S_te.astype(np.float32), secs=secs)

def load_run(name):
    folds = [k for k in range(N_FOLDS) if os.path.exists(fold_path(name, k))]
    if len(folds) < N_FOLDS:
        raise FileNotFoundError(name)
    oof = np.zeros((len(dev_idx), L), dtype=np.float32)
    ho, te, secs = [], [], 0.0
    for k in folds:
        z = np.load(fold_path(name, k))
        oof[FOLDS[k][1]] = z["S_va"]
        ho.append(z["S_ho"]); te.append(z["S_te"]); secs += float(z["secs"])
    cfg_path = os.path.join(run_dir(name), "config.json")
    cfg = json.load(open(cfg_path)) if os.path.exists(cfg_path) else {}
    return {"name": name, "folds": folds, "oof": oof, "ho": np.mean(ho, 0), "te": np.mean(te, 0),
            "secs": secs, "cfg": cfg}

def evaluate_run(R):
    P = decode_flat(R["oof"], crossfit_thresholds(Y_dev, R["oof"], FOLD_ID))
    out = compute_metrics(Y_dev, P)
    out["f1_std"] = float(np.std([f1_score(Y_dev[FOLD_ID == g], P[FOLD_ID == g], average="macro", zero_division=0)
                                  for g in range(N_FOLDS)]))
    return out

def holdout_eval(R):
    return compute_metrics(Y_ho, decode_flat(R["ho"], tune_thresholds(Y_dev, R["oof"])))

def table_row(R, combo="", method="individual"):
    m = evaluate_run(R)
    return {"experimento": R["name"], "combinación": combo, "método": method,
            "modelos": len(R["cfg"].get("miembros", {R["name"]: None})), "F1": m["f1"], "±": m["f1_std"],
            "precision": m["precision"], "recall": m["recall"], "accuracy": m["accuracy"],
            "F1 holdout": holdout_eval(R)["f1"]}

### Combinadores
Las puntuaciones de cada modelo y clase se estandarizan con su media y desviación *out-of-fold*. Los combinadores que aprenden de las etiquetas se ajustan por *cross-fitting*: las puntuaciones de cada partición salen de un combinador ajustado con las otras dos.

In [7]:
def best_f1_per_class(Yt, S):
    o = np.argsort(-S, axis=0, kind="stable")
    tp = np.cumsum(np.take_along_axis(Yt, o, 0), 0)
    k = np.arange(1, len(S) + 1)[:, None]
    return (2 * tp / (k + Yt.sum(0))).max(0)

def fit_mean(Z, Yt):
    return np.full(Z.shape[1], 1 / Z.shape[1])

def fit_greedy(Z, Yt):
    # Selección voraz (Caruana et al., 2004): se añade con repetición el modelo que más sube la F1 macro
    M = Z.shape[1]
    counts, total = np.zeros(M), np.zeros(Z[:, 0].shape, np.float32)
    best_f, best_counts = -1.0, None
    for it in range(GREEDY_ITERS):
        f = [best_f1_per_class(Yt, (total + Z[:, m]) / (it + 1)).mean() for m in range(M)]
        m = int(np.argmax(f))
        counts[m] += 1
        total += Z[:, m]
        if f[m] > best_f + 1e-6:
            best_f, best_counts = f[m], counts.copy()
    return best_counts / best_counts.sum()

def apply_weights(w, Z):
    return np.tensordot(Z, w, axes=([1], [0])).astype(np.float32)

def _lr_one(X, y, C):
    return None if y.min() == y.max() else LogisticRegression(C=C, max_iter=2000).fit(X, y)

def fit_stack_clase(Z, Yt):
    return Parallel(n_jobs=-1)(delayed(_lr_one)(Z[:, :, j], Yt[:, j], STACK_C_CLASE) for j in range(L))

def apply_stack_clase(models, Z):
    S = Z.mean(1)
    for j, m in enumerate(models):
        if m is not None:
            S[:, j] = m.decision_function(Z[:, :, j])
    return tie_alias(S)

def fit_stack_todas(Z, Yt):
    X = Z.reshape(len(Z), -1)
    return Parallel(n_jobs=-1)(delayed(_lr_one)(X, Yt[:, j], STACK_C_TODAS) for j in range(L))

def apply_stack_todas(models, Z):
    X, S = Z.reshape(len(Z), -1), Z.mean(1)
    for j, m in enumerate(models):
        if m is not None:
            S[:, j] = m.decision_function(X)
    return tie_alias(S)

def fit_stack_lgbm(Z, Yt):
    from lightgbm import LGBMClassifier
    X = Z.reshape(len(Z), -1)
    return [None if Yt[:, j].min() == Yt[:, j].max() else
            LGBMClassifier(**LGBM_PARAMS, random_state=SEED, n_jobs=-1).fit(X, Yt[:, j]) for j in range(L)]

def apply_stack_lgbm(models, Z):
    X, S = Z.reshape(len(Z), -1), Z.mean(1)
    for j, m in enumerate(models):
        if m is not None:
            S[:, j] = m.predict(X, raw_score=True)
    return tie_alias(S)

COMBINERS = {
    "media":       (fit_mean, apply_weights),
    "ponderada":   (fit_greedy, apply_weights),
    "stack_clase": (fit_stack_clase, apply_stack_clase),
    "stack_todas": (fit_stack_todas, apply_stack_todas),
    "stack_lgbm":  (fit_stack_lgbm, apply_stack_lgbm),
}

In [8]:
def standardize(Rs):
    Z, Zho, Zte = [], [], []
    for R in Rs:
        mu, sd = R["oof"].mean(0), R["oof"].std(0) + 1e-6
        Z.append((R["oof"] - mu) / sd); Zho.append((R["ho"] - mu) / sd); Zte.append((R["te"] - mu) / sd)
    st = lambda A: np.stack(A, 1).astype(np.float32)
    return st(Z), st(Zho), st(Zte)

def combine(method, combo):
    # Las combinaciones ya guardadas en Drive con los mismos miembros se cargan sin recalcular
    name = f"{method}__{combo}"
    Rs = [load_run(n) for n in COMBOS[combo]]
    cfg = {"tipo": method, "combinacion": combo, "miembros": {R["name"]: [R["folds"], round(R["secs"], 1)] for R in Rs},
           "folds": list(range(N_FOLDS))}
    if method == "ponderada": cfg["greedy_iters"] = GREEDY_ITERS
    if method.startswith("stack"): cfg["C"] = [STACK_C_CLASE, STACK_C_TODAS]
    if method == "stack_lgbm": cfg["lgbm"] = LGBM_PARAMS
    cfg = json.loads(json.dumps(cfg, default=str))
    cfg_path = os.path.join(ENS_DIR, name, "config.json")
    if os.path.exists(cfg_path):
        old = json.load(open(cfg_path))
        old.pop("pesos", None)
        if old == cfg:
            return load_run(name)

    t0 = time.time()
    Z, Zho, Zte = standardize(Rs)
    S = np.zeros((len(dev_idx), L), np.float32)
    if method == "votos":
        # Cada modelo decide con sus umbrales; la puntuación es la fracción de modelos que predicen la etiqueta
        V, Mg = np.zeros_like(Z), np.zeros_like(Z)
        Vho, Mho, Vte, Mte = np.zeros_like(Zho), np.zeros_like(Zho), np.zeros_like(Zte), np.zeros_like(Zte)
        for m in range(Z.shape[1]):
            T = crossfit_thresholds(Y_dev, Z[:, m], FOLD_ID)
            V[:, m], Mg[:, m] = decode_flat(Z[:, m], T), Z[:, m] - T
            thr = tune_thresholds(Y_dev, Z[:, m])
            Vho[:, m], Mho[:, m] = decode_flat(Zho[:, m], thr), Zho[:, m] - thr
            Vte[:, m], Mte[:, m] = decode_flat(Zte[:, m], thr), Zte[:, m] - thr
        vote = lambda V, M: V.mean(1) + 1e-3 * np.tanh(M.mean(1))
        S, S_ho, S_te = vote(V, Mg), vote(Vho, Mho), vote(Vte, Mte)
    else:
        fit, apply = COMBINERS[method]
        for g in range(N_FOLDS):
            m = FOLD_ID == g
            S[m] = apply(fit(Z[~m], Y_dev[~m]), Z[m])
        params = fit(Z, Y_dev)
        S_ho, S_te = apply(params, Zho), apply(params, Zte)
        if method in ("media", "ponderada"):
            cfg["pesos"] = {R["name"]: round(float(w), 3) for R, w in zip(Rs, params)}

    os.makedirs(os.path.join(ENS_DIR, name), exist_ok=True)
    for k, (_, va) in enumerate(FOLDS):
        save_fold(name, k, S[va], S_ho, S_te, (time.time() - t0) / N_FOLDS)
    json.dump(cfg, open(cfg_path, "w"), indent=2)
    print(f"[{name}] calculada en {time.time() - t0:.0f} s")
    return load_run(name)

# 3. Modelos individuales
F1 macro *out-of-fold* de cada modelo que entra en alguna combinación.

In [9]:
ALL_MODELS = list(dict.fromkeys(n for c in COMBOS.values() for n in c))
singles = pd.DataFrame([table_row(load_run(n)) for n in ALL_MODELS]).sort_values("F1", ascending=False)
display(singles[["experimento", "F1", "±", "precision", "recall", "F1 holdout"]].round(4))
BEST_SINGLE_F1 = singles["F1"].max()

,experimento,F1,±,precision,recall,F1 holdout
3,QW_A1_posweight,0.3643,0.0018,0.3157,0.4517,0.3879
0,B3_llrd,0.3519,0.0082,0.3054,0.4793,0.3665
1,A1_posweight,0.3505,0.0043,0.3085,0.4297,0.3651
4,SB_B1_lora,0.3488,0.0068,0.3112,0.4519,0.3595
6,E3_specter2,0.3481,0.0055,0.2999,0.4758,0.3497
7,A2_asl,0.3465,0.0087,0.3002,0.4373,0.3624
8,B1_lora,0.3444,0.0080,0.2911,0.4752,0.3686
2,SVM_ref,0.3288,0.0049,0.2832,0.4406,0.3277
9,EMB_specter2_C0.1,0.3134,0.0033,0.2707,0.4129,0.3331
5,KNN_specter2,0.3132,0.0040,0.2728,0.4215,0.3229


# 4. Combinaciones
Seis métodos aplicados a cinco grupos de modelos (30 combinaciones).

In [10]:
ENS = {}
rows = []
for combo in COMBOS:
    for method in METHODS:
        R = combine(method, combo)
        ENS[R["name"]] = R
        rows.append(table_row(R, combo, method))
ens_df = pd.DataFrame(rows)
ens_df.to_csv(os.path.join(ENS_OUT, "resultados_ensembles_cv.csv"), index=False)

tabla_51 = ens_df.pivot(index="método", columns="combinación", values="F1").loc[METHODS, list(COMBOS)]
print(f"Mejor modelo individual: {BEST_SINGLE_F1:.4f}")
display(tabla_51.round(4))
display(ens_df.sort_values("F1", ascending=False)[["experimento", "modelos", "F1", "±", "F1 holdout"]].head(8).round(4))

[ponderada__specter2_svm] calculada en 40 s
[votos__specter2_svm] calculada en 10 s
[stack_clase__specter2_svm] calculada en 4 s
[stack_todas__specter2_svm] calculada en 19 s
[stack_lgbm__specter2_svm] calculada en 307 s
[ponderada__top3] calculada en 25 s
[votos__top3] calculada en 9 s
[stack_clase__top3] calculada en 4 s
[stack_todas__top3] calculada en 12 s
[stack_lgbm__top3] calculada en 276 s
[stack_todas__top3_svm] calculada en 28 s
[stack_lgbm__top3_svm] calculada en 395 s
[votos__todo] calculada en 29 s
[stack_lgbm__todo] calculada en 1000 s
Mejor modelo individual: 0.3643


combinación,specter2_svm,top3,top3_svm,top6,todo
método,,,,,
media,0.3540,0.3662,0.3623,0.3623,0.3614
ponderada,0.3567,0.3628,0.3628,0.3617,0.3668
votos,0.3547,0.3627,0.3648,0.3621,0.3589
stack_clase,0.3563,0.3640,0.3581,0.3615,0.3597
stack_todas,0.3460,0.3465,0.3471,0.3446,0.3283
stack_lgbm,0.3157,0.3090,0.3169,0.3251,0.3207


,experimento,modelos,F1,±,F1 holdout
25,ponderada__todo,11,0.3668,0.0042,0.3862
6,media__top3,3,0.3662,0.0060,0.3853
14,votos__top3_svm,4,0.3648,0.0055,0.3826
9,stack_clase__top3,3,0.3640,0.0030,0.3882
13,ponderada__top3_svm,4,0.3628,0.0050,0.3767
7,ponderada__top3,3,0.3628,0.0050,0.3767
8,votos__top3,3,0.3627,0.0059,0.3802
18,media__top6,6,0.3623,0.0032,0.3634


In [11]:
pesos = {n: R["cfg"]["pesos"] for n, R in ENS.items() if n.startswith("ponderada")}
display(pd.DataFrame(pesos).T.fillna(0).round(2))

,B3_llrd,A1_posweight,SVM_ref,QW_A1_posweight,SB_B1_lora,KNN_specter2,E3_specter2,A2_asl,B1_lora,EMB_specter2_C0.1,KNN_bge-large
ponderada__specter2_svm,0.50,0.25,0.25,0.00,0.00,0.0,0.0,0.00,0.00,0.0,0.0
ponderada__top3,0.20,0.00,0.00,0.45,0.35,0.0,0.0,0.00,0.00,0.0,0.0
ponderada__top3_svm,0.20,0.00,0.00,0.45,0.35,0.0,0.0,0.00,0.00,0.0,0.0
ponderada__top6,0.20,0.00,0.00,0.45,0.35,0.0,0.0,0.00,0.00,0.0,0.0
ponderada__todo,0.11,0.00,0.00,0.39,0.17,0.0,0.0,0.22,0.11,0.0,0.0


# 5. Envío a Codabench
La combinación con mayor F1 macro *out-of-fold*, con los umbrales ajustados sobre todas sus predicciones de desarrollo.

In [12]:
SUBMIT = ens_df.sort_values("F1", ascending=False)["experimento"].iloc[0]
R = ENS[SUBMIT]
thr = tune_thresholds(Y_dev, R["oof"])
P_te = decode_flat(R["te"], thr)

tag = f"LC02-ens-{SUBMIT.replace('__', '-')}"
sub = test[["titles", "summaries"]].copy()
sub["terms"] = [str(list(t)) for t in mlb.inverse_transform(P_te)]
sub_dir = os.path.join(ENS_OUT, f"sub_{tag}")
os.makedirs(sub_dir, exist_ok=True)
csv_path = os.path.join(sub_dir, "dataset_test_preds.csv")
sub.to_csv(csv_path, index=False)
with zipfile.ZipFile(os.path.join(sub_dir, f"{tag}.zip"), "w", zipfile.ZIP_DEFLATED) as z:
    z.write(csv_path, arcname="dataset_test_preds.csv")
np.save(os.path.join(sub_dir, "test_scores.npy"), R["te"])
np.save(os.path.join(sub_dir, "thresholds.npy"), thr)

assert len(sub) == len(test) and sub["terms"].map(lambda s: len(ast.literal_eval(s)) > 0).all()
print(f"{SUBMIT}: {P_te.sum(1).mean():.2f} etiquetas por artículo en test")

ponderada__todo: 2.25 etiquetas por artículo en test
